# Lab 04 - Natural Language Processing

In this lab, we will cover the following NLP fundamentals:

1. **Tokenization** - Breaking text into subword units
2. **Token Embeddings** - Dense vector representations of tokens
3. **Sentence Embeddings** - Representing entire sentences as vectors
4. **Text Classification** - Classifying documents using embeddings (AG News)
5. **Named Entity Recognition (NER)** - Identifying entities in text
6. **Sequence-to-Sequence (Translation)** - Translating between languages
7. **Language Modeling & Masked Language Modeling** - Predicting tokens in context

### Setup

Run the cell below to install and import all required packages.

In [ ]:
!pip install transformers datasets sentence-transformers scikit-learn torch numpy -q

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel, pipeline
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer

---
## Part 1: Tokenization

Tokenization is the process of breaking text into smaller units called **tokens**. Modern NLP models do not use simple whitespace splitting — they use **subword tokenization** algorithms like Byte-Pair Encoding (BPE) or WordPiece.

This is critical because:
- It handles out-of-vocabulary words by breaking them into known subwords
- It keeps the vocabulary size manageable
- Different models use different tokenizers, which affects how text is represented

### 1.1 Whitespace Tokenization (Naive)

The simplest approach: split on spaces.

In [ ]:
text = "The unhappiness of the tokenizer's creator was unmistakable."

whitespace_tokens = text.split()
print(f"Original: {text}")
print(f"Whitespace tokens ({len(whitespace_tokens)}): {whitespace_tokens}")

Notice that whitespace tokenization doesn't handle punctuation attached to words (e.g., `tokenizer's`, `unmistakable.`) and treats compound/rare words as single opaque tokens.

### 1.2 Subword Tokenization (BPE vs WordPiece)

Modern models use subword tokenizers. Let's compare how **BERT** (WordPiece) and **GPT-2** (BPE) tokenize the same sentence.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "The unhappiness of the tokenizer's creator was unmistakable."

bert_tokens = bert_tokenizer.tokenize(text)
gpt2_tokens = gpt2_tokenizer.tokenize(text)

print(f"Original:     {text}")
print(f"BERT tokens ({len(bert_tokens):2d}):  {bert_tokens}")
print(f"GPT-2 tokens ({len(gpt2_tokens):2d}): {gpt2_tokens}")

Notice how:
- BERT's WordPiece splits `unhappiness` into subwords like `un`, `##happiness` (the `##` prefix means "continuation of previous token")
- GPT-2's BPE uses a different splitting strategy with `Ġ` prefix to denote word boundaries
- Both handle rare/compound words gracefully by decomposing them

### 1.3 Token IDs and Special Tokens

Tokenizers also convert tokens to integer IDs and add special tokens (like `[CLS]` and `[SEP]` for BERT).

In [ ]:
encoded = bert_tokenizer(text, return_tensors="pt")

print(f"Input IDs:      {encoded['input_ids']}")
print(f"Attention Mask: {encoded['attention_mask']}")
print(f"Decoded back:   {bert_tokenizer.decode(encoded['input_ids'][0])}")
print(f"\nSpecial tokens: [CLS] = {bert_tokenizer.cls_token_id}, [SEP] = {bert_tokenizer.sep_token_id}, [PAD] = {bert_tokenizer.pad_token_id}")

**Exercise 1:** Tokenize the following sentence with both BERT and GPT-2 tokenizers. How many tokens does each produce? Why might the counts differ?

```
"Deoxyribonucleic acid (DNA) stores biological information."
```

In [ ]:
# YOUR CODE HERE


---
## Part 2: Token Embeddings

Once we have token IDs, we need to convert them into dense vectors (embeddings). An embedding layer is essentially a lookup table that maps each token ID to a learnable vector.

### 2.1 Building an Embedding Layer from Scratch

In [ ]:
# Build a small vocabulary from a corpus
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog"
]

# Create vocabulary mapping
all_words = set()
for sentence in corpus:
    all_words.update(sentence.split())

word_to_idx = {word: idx for idx, word in enumerate(sorted(all_words))}
print(f"Vocabulary ({len(word_to_idx)} words): {word_to_idx}")

# Create embedding layer
EMBEDDING_DIM = 8
embedding_layer = nn.Embedding(len(word_to_idx), EMBEDDING_DIM)

# Look up embeddings for individual words
for word in ["cat", "dog", "mat"]:
    idx = torch.LongTensor([word_to_idx[word]])
    vec = embedding_layer(idx)
    print(f"'{word}' (id={word_to_idx[word]}): {vec.detach().numpy().flatten()}")

These embeddings are **randomly initialized**. They only become meaningful after training on a task. Let's compare with pretrained embeddings.

### 2.2 Pretrained Token Embeddings from BERT

BERT's embedding layer has been trained on massive text corpora. Let's extract its token embeddings and see how they capture semantic relationships.

In [ ]:
model = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Get embeddings for semantically related words
words = ["king", "queen", "man", "woman", "dog", "cat", "car", "truck"]

word_embeddings = {}
for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the output for the actual word token (index 1, skipping [CLS])
    word_embeddings[word] = outputs.last_hidden_state[0, 1, :]

# Compute cosine similarities between word pairs
cos = nn.CosineSimilarity(dim=0)
pairs = [("king", "queen"), ("man", "woman"), ("king", "dog"), ("cat", "dog"), ("car", "truck"), ("car", "cat")]

print("Cosine Similarities between BERT token embeddings:")
print("-" * 45)
for w1, w2 in pairs:
    sim = cos(word_embeddings[w1], word_embeddings[w2]).item()
    print(f"  {w1:6s} <-> {w2:6s} : {sim:.4f}")

Semantically related words (king-queen, car-truck) should have higher similarity than unrelated words (king-dog, car-cat).

**Exercise 2:** Pick 4 words of your choice and compute the pairwise cosine similarities using the BERT embeddings. Do the similarities match your intuition? Explain.

In [ ]:
# YOUR CODE HERE


---
## Part 3: Sentence Embeddings

A sentence embedding is a single vector that represents an entire sentence. There are multiple ways to compute one.

### 3.1 Why Naive Averaging is Problematic

A common mistake is to simply average all token embeddings to get a sentence vector. The problem: this ignores **padding tokens** and **special tokens** (`[CLS]`, `[SEP]`), which dilute the representation.

When sentences have different lengths, padding tokens (zeros) get averaged in, meaning the resulting vector is biased by sentence length rather than content.

In [ ]:
# Demonstrate the problem with naive averaging
sentences = [
    "I love this movie",
    "I love this movie so much it is truly the best film I have ever seen in my entire life"
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Tokenize with padding (as you would in a batch)
encoded = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**encoded)
    hidden_states = outputs.last_hidden_state  # shape: [batch, seq_len, hidden_dim]

# WRONG: Naive average over ALL positions (including padding and special tokens)
naive_embeddings = hidden_states.mean(dim=1)  # shape: [batch, hidden_dim]

cos = nn.CosineSimilarity(dim=0)
naive_sim = cos(naive_embeddings[0], naive_embeddings[1]).item()
print(f"Naive averaging similarity: {naive_sim:.4f}")
print(f"Attention mask:\n{encoded['attention_mask']}")
print(f"\nNotice sentence 1 has {encoded['attention_mask'][0].sum().item():.0f} real tokens but {encoded['attention_mask'].shape[1]} positions (rest are padding)")

### 3.2 Proper Mean Pooling (Attention-Mask Weighted)

The correct approach is to **mask out padding tokens** before averaging. We use the attention mask to zero out padding positions, then divide by the number of real tokens.

In [ ]:
def mean_pooling(model_output, attention_mask):
    """Compute mean of token embeddings, weighted by attention mask to ignore padding."""
    token_embeddings = model_output.last_hidden_state  # [batch, seq_len, hidden]
    # Expand attention mask to match embedding dimensions
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()  # [batch, seq_len, hidden]
    # Sum embeddings for real tokens only, then divide by count of real tokens
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)  # [batch, hidden]
    sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)  # [batch, hidden]
    return sum_embeddings / sum_mask

proper_embeddings = mean_pooling(outputs, encoded['attention_mask'])

proper_sim = cos(proper_embeddings[0], proper_embeddings[1]).item()
print(f"Naive averaging similarity:  {naive_sim:.4f}")
print(f"Proper mean pooling similarity: {proper_sim:.4f}")

### 3.3 CLS Token Embedding

BERT was trained with a special `[CLS]` token at position 0. Its final hidden state is designed to capture the overall sentence meaning (especially for classification tasks).

In [ ]:
cls_embeddings = hidden_states[:, 0, :]  # [CLS] is always at position 0
cls_sim = cos(cls_embeddings[0], cls_embeddings[1]).item()

print(f"CLS token similarity: {cls_sim:.4f}")

### 3.4 Sentence Transformers (Purpose-Built Sentence Embeddings)

**Sentence Transformers** are models specifically fine-tuned to produce high-quality sentence embeddings. They are far superior to using raw BERT for sentence-level tasks.

In [ ]:
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Compare several sentence pairs
sentence_pairs = [
    ("I love this movie", "This film is amazing"),
    ("I love this movie", "I hate this movie"),
    ("I love this movie", "The weather is sunny today"),
    ("The cat sat on the mat", "A feline rested on the rug"),
]

print("Sentence Transformer Similarities:")
print("-" * 65)
for s1, s2 in sentence_pairs:
    embs = st_model.encode([s1, s2])
    sim = cos(torch.tensor(embs[0]), torch.tensor(embs[1])).item()
    print(f"  {sim:.4f}  |  '{s1}' <-> '{s2}'")

**Exercise 3:** Come up with 3 sentence pairs: one with high similarity, one with moderate, and one with low. Compute their similarity using the Sentence Transformer. Do the scores match your expectations?

In [ ]:
# YOUR CODE HERE


---
## Part 4: Text Classification (AG News)

Now we put embeddings to work. We will classify news articles from the **AG News** dataset into 4 categories:
- 0: World
- 1: Sports
- 2: Business
- 3: Sci/Tech

### Strategy
1. **Baseline:** TF-IDF features + Logistic Regression
2. **Pretrained Embeddings:** Sentence Transformer embeddings + Logistic Regression
3. **Goal:** Beat the baseline using pretrained embeddings

### 4.1 Load the Dataset

In [ ]:
dataset = load_dataset("ag_news")

# Use a subset for speed (feel free to increase for better results)
N_TRAIN = 5000
N_TEST = 1000

train_texts = dataset["train"]["text"][:N_TRAIN]
train_labels = dataset["train"]["label"][:N_TRAIN]
test_texts = dataset["test"]["text"][:N_TEST]
test_labels = dataset["test"]["label"][:N_TEST]

label_names = ["World", "Sports", "Business", "Sci/Tech"]

print(f"Training samples: {len(train_texts)}")
print(f"Test samples: {len(test_texts)}")
print(f"\nExample:")
print(f"  Text: {train_texts[0][:100]}...")
print(f"  Label: {label_names[train_labels[0]]}")

### 4.2 Baseline: TF-IDF + Logistic Regression

In [ ]:
# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

# Logistic Regression
clf_baseline = LogisticRegression(max_iter=1000, random_state=42)
clf_baseline.fit(X_train_tfidf, train_labels)

baseline_preds = clf_baseline.predict(X_test_tfidf)
baseline_acc = accuracy_score(test_labels, baseline_preds)

print(f"Baseline Accuracy (TF-IDF + LogReg): {baseline_acc:.4f}")
print(f"\n{classification_report(test_labels, baseline_preds, target_names=label_names)}")

### 4.3 Pretrained Embeddings: Sentence Transformer + Logistic Regression

Now let's use a pretrained Sentence Transformer to produce embeddings, then train the same Logistic Regression on those. The goal is to **beat the baseline**.

In [ ]:
st_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding training set...")
X_train_emb = st_model.encode(train_texts, show_progress_bar=True, batch_size=64)
print("Encoding test set...")
X_test_emb = st_model.encode(test_texts, show_progress_bar=True, batch_size=64)

print(f"Embedding shape: {X_train_emb.shape}")

In [ ]:
clf_emb = LogisticRegression(max_iter=1000, random_state=42)
clf_emb.fit(X_train_emb, train_labels)

emb_preds = clf_emb.predict(X_test_emb)
emb_acc = accuracy_score(test_labels, emb_preds)

print(f"Baseline Accuracy (TF-IDF):                {baseline_acc:.4f}")
print(f"Sentence Transformer Accuracy:             {emb_acc:.4f}")
print(f"Improvement:                               {emb_acc - baseline_acc:+.4f}")
print(f"\n{classification_report(test_labels, emb_preds, target_names=label_names)}")

**Exercise 4:** Try a different Sentence Transformer model (e.g., `"all-mpnet-base-v2"`, `"paraphrase-MiniLM-L6-v2"`) or increase `N_TRAIN`. Can you beat the score above? Report your best accuracy and which model/settings you used.

In [ ]:
# YOUR CODE HERE


---
## Part 5: Named Entity Recognition (NER)

NER identifies and classifies named entities in text into predefined categories such as:
- **PER** - Person names
- **ORG** - Organizations
- **LOC** - Locations
- **MISC** - Miscellaneous entities

We'll use a pretrained NER model from HuggingFace.

In [ ]:
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

texts = [
    "Barack Obama was the 44th president of the United States and lived in Washington D.C.",
    "Apple Inc. was founded by Steve Jobs in Cupertino, California.",
    "The European Union held a summit in Brussels to discuss trade with China.",
]

for text in texts:
    print(f"\nText: {text}")
    entities = ner_pipeline(text)
    for ent in entities:
        print(f"  {ent['entity_group']:5s} | {ent['word']:25s} | confidence: {ent['score']:.3f}")

**Exercise 5:** Write 2 sentences of your own that contain at least 3 different entity types. Run them through the NER pipeline and comment on the results. Did the model miss anything or get anything wrong?

In [ ]:
# YOUR CODE HERE


---
## Part 6: Sequence-to-Sequence (Translation)

Seq2seq models take an input sequence and produce an output sequence. A classic application is **machine translation**.

We'll use the **MarianMT** model for English-to-French translation and **T5** for a general seq2seq demonstration.

### 6.1 Translation with MarianMT

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")

sentences_to_translate = [
    "Machine learning is transforming the world.",
    "The weather is beautiful today.",
    "I would like to order a coffee, please.",
    "Natural language processing enables computers to understand human language.",
]

print("English -> French Translation:")
print("-" * 70)
for sent in sentences_to_translate:
    result = translator(sent, max_length=128)
    print(f"  EN: {sent}")
    print(f"  FR: {result[0]['translation_text']}")
    print()

### 6.2 T5 for Seq2Seq Tasks

T5 (Text-to-Text Transfer Transformer) treats every NLP task as a text-to-text problem. You prefix the input with a task description.

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Translation
input_text = "translate English to German: How are you doing today?"
input_ids = t5_tokenizer(input_text, return_tensors="pt").input_ids

with torch.no_grad():
    output_ids = t5_model.generate(input_ids, max_length=50)

print(f"Task:   {input_text}")
print(f"Output: {t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

# Summarization
input_text = "summarize: The United States of America is a country primarily located in North America. It consists of 50 states, a federal district, five major unincorporated territories, and various minor islands. The country has the world's third-largest land area and third-largest total area."
input_ids = t5_tokenizer(input_text, return_tensors="pt").input_ids

with torch.no_grad():
    output_ids = t5_model.generate(input_ids, max_length=50)

print(f"\nTask:   summarize: The United States of America is a country...")
print(f"Output: {t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

**Exercise 6:** Use T5 to translate a sentence from English to French (prefix: `"translate English to French: "`). Compare the T5 output with the MarianMT output from 6.1. Which is better?

In [ ]:
# YOUR CODE HERE


---
## Part 7: Language Modeling (LM) & Masked Language Modeling (MLM)

Language models learn to predict tokens in context. There are two main paradigms:

- **Causal LM (autoregressive):** Predict the *next* token given all previous tokens. (GPT-2)
- **Masked LM (MLM):** Predict *masked* tokens given the surrounding context. (BERT)

### 7.1 Causal Language Modeling with GPT-2

GPT-2 generates text by predicting one token at a time, left to right.

In [ ]:
generator = pipeline("text-generation", model="gpt2")

prompts = [
    "Artificial intelligence will",
    "The future of natural language processing is",
    "In a world where robots",
]

print("GPT-2 Text Generation (Causal LM):")
print("=" * 70)
for prompt in prompts:
    result = generator(prompt, max_new_tokens=30, num_return_sequences=1, do_sample=True, temperature=0.7)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {result[0]['generated_text']}")

### 7.2 Masked Language Modeling with BERT

BERT was trained by masking 15% of tokens and learning to predict them from context. This is bidirectional — it can use both left and right context.

In [ ]:
mlm_pipeline = pipeline("fill-mask", model="bert-base-uncased")

masked_sentences = [
    "The capital of France is [MASK].",
    "I went to the [MASK] to buy groceries.",
    "She is a [MASK] at the university.",
    "The [MASK] orbits around the sun.",
]

print("BERT Masked Language Modeling:")
print("=" * 70)
for sentence in masked_sentences:
    results = mlm_pipeline(sentence)
    print(f"\nInput: {sentence}")
    print("Top 5 predictions:")
    for r in results[:5]:
        print(f"  {r['score']:.4f}  ->  {r['token_str']:15s}  |  {r['sequence']}")

### 7.3 Comparing LM vs MLM

Let's highlight the key difference: GPT-2 can only use **left context**, while BERT uses **both directions**.

In [ ]:
# BERT can use right context to disambiguate
print("BERT (bidirectional) - uses context from BOTH sides:")
for sent in ["The [MASK] barked at the mailman.", "The [MASK] meowed at the mailman."]:
    results = mlm_pipeline(sent)
    top = results[0]
    print(f"  '{sent}' -> '{top['token_str']}' ({top['score']:.3f})")

print("\nGPT-2 (left-to-right) - can only continue from the left:")
for prompt in ["The animal barked", "The animal meowed"]:
    result = generator(prompt, max_new_tokens=10, num_return_sequences=1, do_sample=False)
    print(f"  '{prompt}' -> '{result[0]['generated_text']}'")

**Exercise 7:** Create 3 masked sentences where the correct word depends heavily on the **right context** (i.e., words after the mask). Run them through the BERT MLM pipeline and verify that BERT uses the right context effectively.

In [ ]:
# YOUR CODE HERE


---
## Summary

In this lab we covered:

| Topic | Key Takeaway |
|---|---|
| **Tokenization** | Subword tokenizers (BPE, WordPiece) handle rare words gracefully |
| **Token Embeddings** | Pretrained embeddings (BERT) capture semantic relationships |
| **Sentence Embeddings** | Use attention-mask pooling or Sentence Transformers, not naive averaging |
| **Text Classification** | Pretrained embeddings + simple classifiers can beat TF-IDF baselines |
| **NER** | Pretrained token classifiers identify entities with high accuracy |
| **Seq2Seq / Translation** | MarianMT and T5 handle translation; T5 treats all tasks as text-to-text |
| **LM vs MLM** | Causal LM (GPT-2) generates left-to-right; MLM (BERT) uses bidirectional context |